In [ ]:

import numpy as np
import pandas as pd


In [3]:
"""IMPORT DES DATAFRAMES DETAIL : 
df_best_param : pas utilité
df_in : prédictions in-sample
df_oos : prédictions out-of-sample 
df_r2_split_oos : r2 de chaque split """

df_best_param = pd.read_excel("df_best_param.xlsx")
df_in = pd.read_excel("df_in.xlsx")
df_oos = pd.read_excel("df_oos.xlsx")
df_r2_split = pd.read_excel("df_r2_split.xlsx")

MESURES R² OOS IN SAMPLE BENCHMARK

In [4]:
"""FONCTIONS METRIQUES :
R² : calcul du R² selon le papier de Gu et al. (2020) 
% ratio : détermine si le modèle prédit bien le bon signe de la prédiction 
R2 benchmark : MSEmodel / MSEha (Xiu and Liu 2024)
"""
#R²
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

In [5]:
#Tableau in sample contient : R² in sample, Success ratio in sample, comparaison avec HA 

predictions_in = {
    "OLS": df_in["y_trainval_ols"],
    "PLS": df_in["y_trainval_pls"],
    "PCR": df_in["y_trainval_pcr"],
    "Enet": df_in["y_trainval_en"],
    "RF": df_in["y_trainval_rf"],
    "GBRT": df_in["y_trainval_gbrt"],
    "XGB": df_in["y_trainval_xgb"],
    "HA": df_in["y_trainval_pred_ha"]
}

y_trainval_pred_ha = df_in["y_trainval_pred_ha"]
y_trainval_true = df_in["y_trainval_true"]

#TABLE 1 ET 2 : R², success ratio, comparaison HA
rows = []
for model_name, y_pred in predictions_in.items():
    r2_vs = r2_vs_benchmark(y_trainval_true, y_pred, y_trainval_pred_ha)
    r2_in = r2(y_trainval_true, y_pred)
    sr_in = success_ratio(y_trainval_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "In-sample $R^2$": r2_in,
        "Success Ratio": sr_in,
        "$R^2$ vs HA" : r2_vs })
    
df_r2_in_vs = pd.DataFrame(rows)

# Pivot simple pour transformer les splits en colonnes

print(df_r2_in_vs)
#print(df_r2_in_vs)
#print(df_pivot)

##Conversion → Latex 
df_latex_r2in = df_r2_in_vs.copy()
cols_to_convert = ["$R^2$ vs HA", "In-sample $R^2$", "Success Ratio"]

for col in cols_to_convert:
    df_latex_r2in[col] = (df_latex_r2in[col] * 100).apply(lambda x: f"{x:.2f}") #* 100 et arrondir trois chiffres après la virgule

latex_table = df_latex_r2in.to_latex(index=False, escape=False)
print(latex_table)

  Model  In-sample $R^2$  Success Ratio  $R^2$ vs HA
0   OLS         0.025740       0.563448     0.005831
1   PLS         0.023241       0.565312     0.003281
2   PCR         0.021439       0.565482     0.001442
3  Enet         0.023675       0.564804     0.003724
4    RF         0.075681       0.572685     0.056793
5  GBRT         0.046469       0.575736     0.026984
6   XGB         0.044920       0.570143     0.025403
7    HA         0.020025       0.565821     0.000000
\begin{tabular}{llll}
\toprule
Model & In-sample $R^2$ & Success Ratio & $R^2$ vs HA \\
\midrule
OLS & 2.57 & 56.34 & 0.58 \\
PLS & 2.32 & 56.53 & 0.33 \\
PCR & 2.14 & 56.55 & 0.14 \\
Enet & 2.37 & 56.48 & 0.37 \\
RF & 7.57 & 57.27 & 5.68 \\
GBRT & 4.65 & 57.57 & 2.70 \\
XGB & 4.49 & 57.01 & 2.54 \\
HA & 2.00 & 56.58 & 0.00 \\
\bottomrule
\end{tabular}



In [6]:
#Calculs métriques out-of-sample

#Prédictions out of sample 
y_true = df_oos["y_true"]

predictions_oos = {
    "OLS": df_oos["y_pred_ols"],
    "PLS": df_oos["y_pred_pls"],
    "PCR": df_oos["y_pred_pcr"],
    "Enet": df_oos["y_pred_en"],
    "RF": df_oos["y_pred_rf"],
    "GBRT": df_oos["y_pred_gbrt"],
    "XGB": df_oos["y_pred_xgb"],
    "HA": df_oos["y_pred_ha"]
}

#Calcul de R²
rows = []
for model_name, y_pred in predictions_oos.items():
    r2_oos = r2(y_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "Out-of-sample $R^2$": r2_oos,})
df_r2_oos = pd.DataFrame(rows)

#R² par split  

df_r2_split_oos = df_r2_split[["Model", "Split 1 o", "Split 2 o", "Split 3 o"]].copy()
df_r2_split_oos.rename(columns={"Split 1 o":  "Split 1", "Split 2 o": "Split 2", "Split 3 o": "Split 3"}, inplace=True)

df_r2_oos = df_r2_split_oos.merge(
    df_r2_oos, on="Model", how="inner")

df_latex_r2_oos = df_r2_oos.copy()
cols_to_convert = ["Split 1", "Split 2", "Split 3", "Out-of-sample $R^2$"]

for col in cols_to_convert:
    df_latex_r2_oos[col] = (df_latex_r2_oos[col] * 100).apply(lambda x: f"{x:.2f}")

latex_table = df_latex_r2_oos.to_latex(index=False, escape=False)
print(latex_table)


\begin{tabular}{lllll}
\toprule
Model & Split 1 & Split 2 & Split 3 & Out-of-sample $R^2$ \\
\midrule
HA & 2.14 & 2.78 & 0.75 & 1.79 \\
OLS & 2.20 & 3.18 & 0.12 & 1.68 \\
PLS & 2.74 & 2.91 & 0.29 & 1.76 \\
PCR & 2.40 & 2.54 & 0.15 & 1.50 \\
RF & 2.94 & 3.15 & 1.66 & 2.47 \\
GBRT & 3.71 & 3.74 & 0.64 & 2.41 \\
XGB & 3.43 & 3.86 & 1.40 & 2.73 \\
\bottomrule
\end{tabular}



In [ ]:
from scipy.stats import t

models = [col for col in df_oos.columns if col.startswith('y_pred_')]
results = []

for i in range(len(models)):
    for j in range(i+1, len(models)):
        model1 = models[i]
        model2 = models[j]

        #calcule mse du modèle 1 et 2 
        e1 = (df_oos['y_true'] - df_oos[model1])**2
        e2 = (df_oos['y_true'] - df_oos[model2])**2
        d = e1 - e2
        d = d.dropna()

        X = np.ones(len(d))  # régression constante
        model = sm.OLS(d, X).fit()
        cov = cov_hac(model, nlags=1)  
        se = np.sqrt(cov[0][0])
        dm_stat = d.mean() / se
        p_value = 2 * (1 - t.cdf(abs(dm_stat), df=len(d) - 1))

        results.append({
            'Model 1': model1,
            'Model 2': model2,
            'DM Stat': dm_stat,
            'P-Value': p_value,
            'Best Model': model2 if dm_stat > 0 else model1
        })


dm_df = pd.DataFrame(results)
dm_df.sort_values(by="P-Value", ascending=True, inplace=True)
print(dm_df)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd

# (Optionnel) Style cohérent avec LaTeX
matplotlib.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Palatino"],
    "figure.figsize": (6.2, 4.2),  # ≈ \textwidth, pour Overleaf
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9
})

# Exemple : df_best_param = pd.read_csv("best_params.csv")
# Ton dataframe doit contenir les colonnes : 'Date', 'Best lambda', 'Best_k_pcr', ...

fig, axes = plt.subplots(2, 3, figsize=(6.2, 4), sharex=True)
fig.suptitle(r"\textbf{Time-varying Model Complexity}", fontsize=11)

model_params = [
    (r"ENet+H", "Best lambda", r"\# of Char."),
    (r"PCR", "Best_k_pcr", r"\# of Comp."),
    (r"PLS", "Best_k_pls", r"\# of Comp."),
    (r"RF", "max_depths_rf", "Tree Depth"),
    (r"GBRT+H", "max_depths_gbrt", r"\# of Char."),
    (r"XGB", "max_depths_xgb", "Tree Depth")
]

for ax, (title, col, ylabel) in zip(axes.flatten(), model_params):
    ax.plot(df_best_param["Date"], df_best_param[col])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Year")
    ax.grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# Export PDF (vectoriel, qualité Overleaf)
plt.savefig("model_complexity.pdf", bbox_inches="tight")


PORTEFEUILLES

In [ ]:
df_me = pd.read_excel("me_df.xlsx")
df_me = df_me.sort_values("Date").reset_index(drop=True) 

In [ ]:
df_predict = df_oos.copy()

In [ ]:
#I. PORTEFEUILLES EQUALLY WEIGHT

pred_cols = ['y_pred_ols','y_pred_pls','y_pred_pcr','y_pred_en',
             'y_pred_rf','y_pred_gbrt','y_pred_xgb','y_pred_ha']

col_ret = 'y_true'
nb_groups = 3  

# Résultats finaux
final_table = {}

for col_pred in pred_cols:
    all_rows = []

    for date, i in df_predict.groupby('Date'):
        i = i.sort_values(col_pred).reset_index(drop=True)
        n = len(i)
        size = n // nb_groups
        groups = [min(idx // size, nb_groups - 1) for idx in range(n)]
        i["group"] = groups

        for grp in range(nb_groups):
            sub = i[i["group"] == grp]
            mean_pred = sub[col_pred].mean()
            mean_real = sub[col_ret].mean()
            std_real = sub[col_ret].std()
            sr_real = mean_real / std_real if std_real != 0 else np.nan

            all_rows.append({
                "Group": grp,
                "Pred": mean_pred,
                "Avg": mean_real,
                "SD": std_real,
                "SR": sr_real
            })

    # Convertir en DataFrame
    df_result = pd.DataFrame(all_rows)
    df_grouped = df_result.groupby("Group").mean().reset_index()

    # Ajouter H-L
    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg":  df_grouped.loc[nb_groups-1, "Avg"]  - df_grouped.loc[0, "Avg"],
        "SD":   df_grouped.loc[nb_groups-1, "SD"],  # ou recompute spread SD
        "SR":   (df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"]) / df_grouped["SD"].mean()
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table[col_pred.upper()] = df_grouped

# 🔁 Tu peux afficher tous les tableaux :
for model, df in final_table.items():
    print(f"\n==== {model} ====")
    print(df.to_string(index=False))

In [ ]:
# Associe la market cap à df_predict
#df_predict = df_predict.merge(df_me[['Date', 'Ticker', 'Mkt_Cap_Monthly']], on=['Date', 'Ticker'], how='left')

In [ ]:


col_ret = 'y_true'
col_weight = 'Mkt_Cap_Monthly'
nb_groups = 3

final_table_vw = {}

for col_pred in pred_cols:
    all_rows = []

    for date, i in df_predict.groupby('Date'):
        i = i.sort_values(col_pred).reset_index(drop=True)
        i[col_weight] = i[col_weight].fillna(0)

        n = len(i)
        size = n // nb_groups
        groups = [min(idx // size, nb_groups - 1) for idx in range(n)]
        i["group"] = groups

        for grp in sorted(i["group"].unique()):
            sub = i[i["group"] == grp]
            w = sub[col_weight]
            if w.sum() == 0:
                continue  # éviter division par zéro

            w_norm = w / w.sum()
            pred_mean = np.average(sub[col_pred], weights=w_norm)
            ret_mean = np.average(sub[col_ret], weights=w_norm)
            ret_std = np.sqrt(np.average((sub[col_ret] - ret_mean)**2, weights=w_norm))
            ret_sr = ret_mean / ret_std if ret_std != 0 else np.nan

            all_rows.append({
                "Group": grp,
                "Pred": pred_mean,
                "Avg": ret_mean,
                "SD": ret_std,
                "SR": ret_sr
            })

    df_result = pd.DataFrame(all_rows)
    df_grouped = df_result.groupby("Group").mean().reset_index()

    # Ajouter ligne H-L
    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg": df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"],
        "SD": df_grouped.loc[nb_groups-1, "SD"],  # facultatif : recompute spread SD
        "SR": (df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"]) / df_grouped["SD"].mean()
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table_vw[col_pred.upper()] = df_grouped

# 🔁 Affichage
for model, df in final_table_vw.items():
    print(f"\n==== {model} (VW) ====")
    print(df.to_string(index=False))
